In [1]:
# ============================================================
# 🎬 Netflix Churn Prediction — KERAS (PIPELINE-COMPATIBLE)
# Production-grade • Fairness-ready • InferStream-compatible
# ============================================================

import pandas as pd
import numpy as np
import os
import json
import joblib
import datetime

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# ================================
# Paths
# ================================
DATA_PATH = "../data/netflix_customer_churn.csv"
MODEL_DIR = "../backend/models/netflix"

MODEL_NAME = "keras_model.h5"
SCALER_NAME = "scaler_keras.pkl"
ENCODER_NAME = "encoder_keras.pkl"
FEATURES_NAME = "feature_names_keras.json"

os.makedirs(MODEL_DIR, exist_ok=True)

# ================================
# Load Dataset
# ================================
df = pd.read_csv(DATA_PATH)
print("✅ Loaded dataset:", df.shape)

TARGET_COL = "churned"

df = df.dropna(subset=[TARGET_COL])
df[TARGET_COL] = df[TARGET_COL].astype(int)

# Drop identifiers
for col in ["customer_id", "email", "user_id"]:
    if col in df.columns:
        df = df.drop(columns=[col])

y = df[TARGET_COL]
X = df.drop(columns=[TARGET_COL])

# ================================
# Feature Groups
# ================================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("🔢 Numeric features:", num_cols)
print("🏷️ Categorical features:", cat_cols)

# ================================
# Preprocessing (SKLEARN)
# ================================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ]
)

# ================================
# Transform Features
# ================================
X_processed = preprocessor.fit_transform(X)

feature_names = preprocessor.get_feature_names_out().tolist()

# ================================
# Train / Test Split
# ================================
X_train, X_test, y_train, y_test = train_test_split(
    X_processed,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

print(f"📊 Train samples: {X_train.shape}")
print(f"📊 Test samples:  {X_test.shape}")

# ================================
# Build Keras Model
# ================================
model = Sequential([
    Dense(64, activation="relu", input_dim=X_train.shape[1]),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dense(y_train_cat.shape[1], activation="softmax"),
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# ================================
# Train
# ================================
model.fit(
    X_train,
    y_train_cat,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test_cat),
    verbose=1
)

# ================================
# Evaluate
# ================================
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("🧩 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ================================
# Save Artifacts (CRITICAL)
# ================================
model.save(os.path.join(MODEL_DIR, MODEL_NAME))

joblib.dump(
    preprocessor.named_transformers_["num"],
    os.path.join(MODEL_DIR, SCALER_NAME)
)

joblib.dump(
    preprocessor.named_transformers_["cat"],
    os.path.join(MODEL_DIR, ENCODER_NAME)
)

with open(os.path.join(MODEL_DIR, FEATURES_NAME), "w") as f:
    json.dump(feature_names, f, indent=2)

print("✅ Saved artifacts:")
print(f"   • {MODEL_NAME}")
print(f"   • {SCALER_NAME}")
print(f"   • {ENCODER_NAME}")
print(f"   • {FEATURES_NAME}")

print("🏁 Done at", datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

✅ Loaded dataset: (5000, 14)
🔢 Numeric features: ['age', 'watch_hours', 'last_login_days', 'monthly_fee', 'number_of_profiles', 'avg_watch_time_per_day']
🏷️ Categorical features: ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']
📊 Train samples: (4000, 35)
📊 Test samples:  (1000, 35)


/var/folders/91/syqbgxdn69n01td9pvbx3dz80000gn/T/ipykernel_80643/1001091392.py:58: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
/Users/darrylcarp/Coding/VScodeProjects/INFERSTREAM/.venv64/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **k

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            66 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,450 (17.38 KB)

 Trainable params: 4,450 (17.38 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.7905 - loss: 0.4401 - val_accuracy: 0.8690 - val_loss: 0.2720
Epoch 2/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 619us/step - accuracy: 0.8748 - loss: 0.2846 - val_accuracy: 0.8870 - val_loss: 0.2349
Epoch 3/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 617us/step - accuracy: 0.8840 - loss: 0.2608 - val_accuracy: 0.8910 - val_loss: 0.2289
Epoch 4/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 684us/step - accuracy: 0.8915 - loss: 0.2512 - val_accuracy: 0.9000 - val_loss: 0.2208
Epoch 5/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 694us/step - accuracy: 0.9018 - loss: 0.2335 - val_accuracy: 0.8950 - val_loss: 0.2197
Epoch 6/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 673us/step - accuracy: 0.8992 - loss: 0.2310 - val_accuracy: 0.9020 - val_loss: 0.2129
Epoch 7/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 642us/step - accuracy: 0.9018 - loss: 0.2225 - val_accuracy: 0.9090 - val_loss: 0.2092
Epoch 8/15
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 624us/step - accuracy: 0.9072 - loss: 0.2187 - va


📋 Classification Report:
              precision    recall  f1-score   support

           0     0.9106    0.9014    0.9060       497
           1     0.9035    0.9125    0.9080       503

    accuracy                         0.9070      1000
   macro avg     0.9071    0.9070    0.9070      1000
weighted avg     0.9070    0.9070    0.9070      1000

🧩 Confusion Matrix:
[[448  49]
 [ 44 459]]
✅ Saved artifacts:
   • keras_model.h5
   • scaler_keras.pkl
   • encoder_keras.pkl
   • feature_names_keras.json
🏁 Done at 2026-01-24 18:46:23
